In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# ForensicNLP — Notebook 01: Data Extraction and Dataset Construction
====================================================================
This notebook reads the MITRE ATT&CK v18.1 Excel files, extracts
procedure examples, maps them to tactic labels, cleans the text,
and creates train/dev/test splits for the classification task.

HOW TO USE IN GOOGLE COLAB:
1. Upload enterprise-attack-v18_1.xlsx and mobile-attack-v18_1.xlsx
   to your Google Drive under: MyDrive/forensicnlp/data/raw/
2. Open this file in Colab (or paste each section into cells)
3. Run cells in order

Each section marked with


In [ ]:
#is a separate Colab cell.
#"""

# ForensicNLP — Data Extraction & Dataset Construction

This notebook performs the following steps:
1. Mount Google Drive and load ATT&CK Excel files
2. Extract procedure examples from the `relationships` sheet
3. Map each example to its ATT&CK tactic label
4. Clean text (remove citations, markdown links, etc.)
5. Analyze class distribution
6. Create train/dev/test splits (case-level, no leakage)
7. Save everything as CSV files


## CELL 1: Setup and Imports


In [16]:
import pandas as pd
import numpy as np
import re
import os
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive (uncomment in Colab)
from google.colab import drive
drive.mount('/content/drive')

# Set your paths — adjust if your folder structure is different
# For Colab:
BASE_DIR = '/content/drive/MyDrive/forensicnlp'
# For local testing:
# BASE_DIR = '.'  # Change this to your local path

RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
CLASS_DIR = os.path.join(BASE_DIR, 'data', 'classification')
TOKEN_DIR = os.path.join(BASE_DIR, 'data', 'tokenization')

# Create output directories
os.makedirs(CLASS_DIR, exist_ok=True)
os.makedirs(TOKEN_DIR, exist_ok=True)

print("✓ Setup complete")
print(f"  Raw data dir:    {RAW_DIR}")
print(f"  Classification:  {CLASS_DIR}")
print(f"  Tokenization:  {TOKEN_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Setup complete
  Raw data dir:    /content/drive/MyDrive/forensicnlp/data/raw
  Classification:  /content/drive/MyDrive/forensicnlp/data/classification
  Tokenization:  /content/drive/MyDrive/forensicnlp/data/tokenization


## CELL 2: Load ATT&CK Excel Files


WHAT THIS DOES:
The ATT&CK Excel files have multiple sheets. We need two:
  - 'techniques': maps technique IDs to tactic names
  - 'relationships': contains procedure examples (how attackers
    actually used each technique, with text descriptions)

We read both enterprise and mobile files.


In [18]:
enterprise_path = os.path.join(RAW_DIR, '/content/drive/MyDrive/forensicnlp/data/raw/enterprise-attack-v18.1.xlsx')
mobile_path = os.path.join(RAW_DIR, '/content/drive/MyDrive/forensicnlp/data/raw/mobile-attack-v18.1.xlsx')

# Check files exist
for path, name in [(enterprise_path, 'Enterprise'), (mobile_path, 'Mobile')]:
    if os.path.exists(path):
        print(f"✓ Found {name}: {path}")
    else:
        print(f"✗ Missing {name}: {path}")
        print(f"  Please upload the file to: {path}")

✓ Found Enterprise: /content/drive/MyDrive/forensicnlp/data/raw/enterprise-attack-v18.1.xlsx
✓ Found Mobile: /content/drive/MyDrive/forensicnlp/data/raw/mobile-attack-v18.1.xlsx


## CELL 3: Build Technique → Tactic Mapping


WHAT THIS DOES:
Each ATT&CK technique (like T1059 "Command and Scripting Interpreter")
belongs to one or more tactics (like "Execution"). We need this mapping
so we can label each procedure example with its tactic.

IMPORTANT: The tactics column is at different positions in each file:
  - Enterprise: column index 9
  - Mobile: column index 10


In [19]:
def load_technique_tactics(filepath, tactics_col_idx):
    """Read the techniques sheet and return a dict: technique_id -> list of tactics."""
    df = pd.read_excel(filepath, sheet_name='techniques')
    cols = df.columns.tolist()
    tactics_col = cols[tactics_col_idx]

    mapping = {}
    for _, row in df.iterrows():
        tech_id = str(row['ID']).strip()
        tactics_raw = str(row[tactics_col]) if pd.notna(row[tactics_col]) else ''
        tactics = [t.strip() for t in tactics_raw.split(',') if t.strip()]
        if tech_id and tactics:
            mapping[tech_id] = tactics
    return mapping

# Enterprise: tactics at column index 9
enterprise_tactics = load_technique_tactics(enterprise_path, 9)
print(f"✓ Enterprise technique→tactic mappings: {len(enterprise_tactics)}")

# Mobile: tactics at column index 10
mobile_tactics = load_technique_tactics(mobile_path, 10)
print(f"✓ Mobile technique→tactic mappings: {len(mobile_tactics)}")

# Combine into one lookup
all_tactics = {**enterprise_tactics, **mobile_tactics}
print(f"✓ Total combined mappings: {len(all_tactics)}")

# Show all unique tactics found
all_tactic_names = set()
for tactics_list in all_tactics.values():
    all_tactic_names.update(tactics_list)
print(f"\nAll {len(all_tactic_names)} unique ATT&CK tactics found:")
for t in sorted(all_tactic_names):
    print(f"  • {t}")

✓ Enterprise technique→tactic mappings: 691
✓ Mobile technique→tactic mappings: 124
✓ Total combined mappings: 815

All 14 unique ATT&CK tactics found:
  • Collection
  • Command and Control
  • Credential Access
  • Defense Evasion
  • Discovery
  • Execution
  • Exfiltration
  • Impact
  • Initial Access
  • Lateral Movement
  • Persistence
  • Privilege Escalation
  • Reconnaissance
  • Resource Development


## CELL 4: Extract Procedure Examples


WHAT THIS DOES:
The 'relationships' sheet contains rows describing how groups, software,
or campaigns USE specific techniques. Each row with:
  - mapping_type == "uses"
  - target_type == "technique"
is a PROCEDURE EXAMPLE — a real-world description of how an attacker
employed a technique. The 'mapping description' column has the text.

These are exactly what we need for classification: text + tactic label.


In [20]:
def extract_procedure_examples(filepath, technique_tactics, domain_label):
    """
    Extract procedure examples from the relationships sheet.
    Returns a list of dicts with: text, technique_id, technique_name,
    tactics, source_name, source_type, domain.
    """
    df = pd.read_excel(filepath, sheet_name='relationships')

    examples = []
    skipped_no_text = 0
    skipped_no_tactic = 0

    for _, row in df.iterrows():
        # Only keep "uses" relationships targeting techniques
        if str(row.get('mapping type', '')) != 'uses':
            continue
        if str(row.get('target type', '')) != 'technique':
            continue

        # Get the description text
        desc = str(row.get('mapping description', '')) if pd.notna(row.get('mapping description')) else ''

        # Clean: remove (Citation: ...) patterns
        desc_clean = re.sub(r'\(Citation:\s*[^)]+\)', '', desc).strip()

        # Clean: convert markdown links [text](url) -> text
        desc_clean = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', desc_clean)

        # Clean: remove leftover parentheses, extra spaces
        desc_clean = re.sub(r'\s+', ' ', desc_clean).strip()

        # Skip if no meaningful text after cleaning
        if len(desc_clean) < 20:
            skipped_no_text += 1
            continue

        # Get technique ID and look up tactics
        tech_id = str(row.get('target ID', '')).strip()
        tactics = technique_tactics.get(tech_id, [])

        if not tactics:
            skipped_no_tactic += 1
            continue

        examples.append({
            'text': desc_clean,
            'technique_id': tech_id,
            'technique_name': str(row.get('target name', '')),
            'tactics': ', '.join(tactics),
            'primary_tactic': tactics[0],  # first listed tactic
            'source_name': str(row.get('source name', '')),
            'source_type': str(row.get('source type', '')),
            'source_id': str(row.get('source ID', '')),
            'domain': domain_label
        })

    print(f"  {domain_label}: {len(examples)} examples extracted, "
          f"{skipped_no_text} skipped (no text), "
          f"{skipped_no_tactic} skipped (no tactic mapping)")

    return examples

print("Extracting procedure examples...")
enterprise_examples = extract_procedure_examples(
    enterprise_path, all_tactics, 'enterprise'
)
mobile_examples = extract_procedure_examples(
    mobile_path, all_tactics, 'mobile'
)

# Combine
all_examples = enterprise_examples + mobile_examples
df_all = pd.DataFrame(all_examples)

print(f"\n✓ Total procedure examples: {len(df_all)}")
print(f"  Enterprise: {len(enterprise_examples)}")
print(f"  Mobile: {len(mobile_examples)}")
print(f"\nSample entry:")
print(f"  Text: {df_all.iloc[0]['text'][:150]}...")
print(f"  Technique: {df_all.iloc[0]['technique_id']} - {df_all.iloc[0]['technique_name']}")
print(f"  Tactics: {df_all.iloc[0]['tactics']}")
print(f"  Source: {df_all.iloc[0]['source_name']} ({df_all.iloc[0]['source_type']})")

Extracting procedure examples...
  enterprise: 16017 examples extracted, 0 skipped (no text), 0 skipped (no tactic mapping)
  mobile: 1404 examples extracted, 0 skipped (no text), 0 skipped (no tactic mapping)

✓ Total procedure examples: 17421
  Enterprise: 16017
  Mobile: 1404

Sample entry:
  Text: During the 2015 Ukraine Electric Power Attack, Sandworm Team modified in-registry internet settings to lower internet security....
  Technique: T1562.001 - Disable or Modify Tools
  Tactics: Defense Evasion
  Source: 2015 Ukraine Electric Power Attack (campaign)


## CELL 5: Analyze Tactic Distribution (Before Mapping)


WHAT THIS DOES:
Before we create our classification labels, let's see what the raw
tactic distribution looks like. This helps us decide how to group
tactics into our final label set.

NOTE: Some techniques map to multiple tactics (e.g., "Defense Evasion,
Privilege Escalation"). For classification, we handle this by using
the PRIMARY tactic (the first one listed by MITRE).


In [21]:
print("=== PRIMARY TACTIC DISTRIBUTION ===\n")
primary_counts = df_all['primary_tactic'].value_counts()
for tactic, count in primary_counts.items():
    bar = '█' * (count // 100)
    print(f"  {tactic:30s} {count:5d}  {bar}")

print(f"\n  {'TOTAL':30s} {len(df_all):5d}")

# Also show multi-tactic frequency
multi_tactic = df_all[df_all['tactics'].str.contains(',')]
print(f"\n{len(multi_tactic)} examples have multiple tactics ({len(multi_tactic)/len(df_all)*100:.1f}%)")

=== PRIMARY TACTIC DISTRIBUTION ===

  Defense Evasion                 4204  ██████████████████████████████████████████
  Discovery                       3216  ████████████████████████████████
  Command and Control             2342  ███████████████████████
  Execution                       2055  ████████████████████
  Collection                      1783  █████████████████
  Persistence                      857  ████████
  Credential Access                622  ██████
  Resource Development             512  █████
  Initial Access                   500  █████
  Impact                           412  ████
  Exfiltration                     406  ████
  Lateral Movement                 282  ██
  Reconnaissance                   155  █
  Privilege Escalation              75  

  TOTAL                          17421

2463 examples have multiple tactics (14.1%)


## CELL 6: Create Classification Label Mapping


WHAT THIS DOES:
The raw ATT&CK data has 14 tactics, but some have very few examples.
For a robust classification experiment, we map them into our research
label scheme.

OUR MAPPING RATIONALE:
- We keep the 8 largest tactical categories as-is
- We group the smallest ones into related categories
- "Benign/Admin" and "Browser/User" will come from supplemental
  data sources later (EVTX logs, Sigma rules)

The mapping below preserves the most forensically meaningful distinctions
while ensuring each class has enough training samples.


In [22]:
# Define our label mapping: ATT&CK tactic → our classification label
TACTIC_TO_LABEL = {
    # Core forensic categories (keep as-is)
    'Defense Evasion':      'Defense Evasion',
    'Discovery':            'Discovery',
    'Execution':            'Execution',
    'Persistence':          'Persistence',
    'Command and Control':  'Command and Control',
    'Credential Access':    'Credential Access',
    'Collection':           'Collection',

    # Group smaller categories into related ones
    'Privilege Escalation': 'Privilege Escalation',
    'Lateral Movement':     'Lateral Movement',
    'Exfiltration':         'Exfiltration',
    'Initial Access':       'Initial Access',
    'Impact':               'Impact',

    # Very small categories → group into nearest large category
    'Reconnaissance':       'Reconnaissance',
    'Resource Development': 'Resource Development',
}

# Apply the mapping
df_all['label'] = df_all['primary_tactic'].map(TACTIC_TO_LABEL)

# Check for any unmapped tactics
unmapped = df_all[df_all['label'].isna()]['primary_tactic'].unique()
if len(unmapped) > 0:
    print(f"⚠ Unmapped tactics: {unmapped}")
    print("  These will be dropped. Update TACTIC_TO_LABEL if needed.")
    df_all = df_all.dropna(subset=['label'])
else:
    print("✓ All tactics mapped successfully")

print(f"\n=== FINAL LABEL DISTRIBUTION ===\n")
label_counts = df_all['label'].value_counts()
for label, count in label_counts.items():
    bar = '█' * (count // 100)
    pct = count / len(df_all) * 100
    print(f"  {label:25s} {count:5d} ({pct:4.1f}%)  {bar}")

print(f"\n  {'TOTAL':25s} {len(df_all):5d}")
print(f"  Number of classes: {df_all['label'].nunique()}")

✓ All tactics mapped successfully

=== FINAL LABEL DISTRIBUTION ===

  Defense Evasion            4204 (24.1%)  ██████████████████████████████████████████
  Discovery                  3216 (18.5%)  ████████████████████████████████
  Command and Control        2342 (13.4%)  ███████████████████████
  Execution                  2055 (11.8%)  ████████████████████
  Collection                 1783 (10.2%)  █████████████████
  Persistence                 857 ( 4.9%)  ████████
  Credential Access           622 ( 3.6%)  ██████
  Resource Development        512 ( 2.9%)  █████
  Initial Access              500 ( 2.9%)  █████
  Impact                      412 ( 2.4%)  ████
  Exfiltration                406 ( 2.3%)  ████
  Lateral Movement            282 ( 1.6%)  ██
  Reconnaissance              155 ( 0.9%)  █
  Privilege Escalation         75 ( 0.4%)  

  TOTAL                     17421
  Number of classes: 14


## CELL 7: Create Case-Level Train/Dev/Test Splits


WHAT THIS DOES:
We split the data into train (70%), dev (15%), and test (15%) sets.

CRITICAL: We do CASE-LEVEL splitting, not random row-level splitting.
This means all procedure examples from the same source (same threat
group, same malware, same campaign) stay in the SAME split.

WHY THIS MATTERS:
If examples from APT29 appear in both train and test, the model could
memorize APT29-specific language patterns instead of learning general
forensic concepts. Case-level splitting prevents this data leakage and
makes the evaluation much more credible.

We use the 'source_id' column as the grouping key.


In [23]:
# Use GroupShuffleSplit to ensure case-level splitting
# First split: 70% train, 30% temp
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(df_all, groups=df_all['source_id']))

df_train = df_all.iloc[train_idx].copy()
df_temp = df_all.iloc[temp_idx].copy()

# Second split: split the 30% into 15% dev, 15% test
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
dev_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['source_id']))

df_dev = df_temp.iloc[dev_idx].copy()
df_test = df_temp.iloc[test_idx].copy()

print("=== SPLIT SIZES ===")
print(f"  Train: {len(df_train):5d} ({len(df_train)/len(df_all)*100:.1f}%)")
print(f"  Dev:   {len(df_dev):5d} ({len(df_dev)/len(df_all)*100:.1f}%)")
print(f"  Test:  {len(df_test):5d} ({len(df_test)/len(df_all)*100:.1f}%)")
print(f"  Total: {len(df_train)+len(df_dev)+len(df_test):5d}")

# Verify no source leakage
train_sources = set(df_train['source_id'])
dev_sources = set(df_dev['source_id'])
test_sources = set(df_test['source_id'])

leak_td = train_sources & dev_sources
leak_tt = train_sources & test_sources
leak_dt = dev_sources & test_sources

print(f"\n=== DATA LEAKAGE CHECK ===")
print(f"  Train ∩ Dev overlap:  {len(leak_td)} sources {'✓ CLEAN' if len(leak_td)==0 else '✗ LEAKAGE!'}")
print(f"  Train ∩ Test overlap: {len(leak_tt)} sources {'✓ CLEAN' if len(leak_tt)==0 else '✗ LEAKAGE!'}")
print(f"  Dev ∩ Test overlap:   {len(leak_dt)} sources {'✓ CLEAN' if len(leak_dt)==0 else '✗ LEAKAGE!'}")

# Show label distribution per split
print(f"\n=== LABEL DISTRIBUTION PER SPLIT ===")
for split_name, split_df in [('Train', df_train), ('Dev', df_dev), ('Test', df_test)]:
    counts = split_df['label'].value_counts()
    print(f"\n  {split_name}:")
    for label, count in counts.items():
        print(f"    {label:25s} {count:4d}")

=== SPLIT SIZES ===
  Train: 12325 (70.7%)
  Dev:    2409 (13.8%)
  Test:   2687 (15.4%)
  Total: 17421

=== DATA LEAKAGE CHECK ===
  Train ∩ Dev overlap:  0 sources ✓ CLEAN
  Train ∩ Test overlap: 0 sources ✓ CLEAN
  Dev ∩ Test overlap:   0 sources ✓ CLEAN

=== LABEL DISTRIBUTION PER SPLIT ===

  Train:
    Defense Evasion           2986
    Discovery                 2278
    Command and Control       1686
    Execution                 1466
    Collection                1277
    Persistence                607
    Credential Access          410
    Resource Development       363
    Initial Access             333
    Exfiltration               287
    Impact                     282
    Lateral Movement           178
    Reconnaissance             123
    Privilege Escalation        49

  Dev:
    Defense Evasion            596
    Discovery                  450
    Command and Control        293
    Execution                  291
    Collection                 206
    Persistence      

## CELL 8: Text Quality Check


WHAT THIS DOES:
Before saving, let's check the quality of our cleaned text.
We want to make sure:
  - No leftover citation markers
  - No empty or near-empty strings
  - Reasonable text lengths
  - Some forensic artifacts are present (paths, commands, etc.)


In [24]:
print("=== TEXT QUALITY STATS ===\n")

# Length statistics
text_lengths = df_all['text'].str.len()
print(f"  Min length:    {text_lengths.min()}")
print(f"  Max length:    {text_lengths.max()}")
print(f"  Mean length:   {text_lengths.mean():.0f}")
print(f"  Median length: {text_lengths.median():.0f}")

# Check for leftover citation markers
has_citation = df_all['text'].str.contains(r'\(Citation:', regex=True).sum()
print(f"\n  Leftover '(Citation:' markers: {has_citation}")

# Check for common forensic string patterns
patterns = {
    'Registry paths (\\)': r'\\[A-Za-z]+\\',
    'File extensions (.exe, .dll)': r'\.\w{2,4}\b',
    'IP addresses': r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}',
    'Hashes (hex 32+ chars)': r'[a-fA-F0-9]{32,}',
    'Command-line tools': r'(cmd\.exe|powershell|wmic|reg\.exe|net\.exe)',
    'ATT&CK technique refs': r'T\d{4}',
}

print(f"\n  Forensic artifact patterns found in text:")
for name, pattern in patterns.items():
    count = df_all['text'].str.contains(pattern, regex=True, case=False).sum()
    pct = count / len(df_all) * 100
    print(f"    {name:40s} {count:5d} ({pct:.1f}%)")

# Show a few example texts
print(f"\n=== SAMPLE TEXTS (5 random examples) ===\n")
for _, row in df_all.sample(5, random_state=42).iterrows():
    print(f"  [{row['label']}]")
    print(f"  {row['text'][:200]}...")
    print()

=== TEXT QUALITY STATS ===

  Min length:    23
  Max length:    517
  Mean length:   96
  Median length: 83

  Leftover '(Citation:' markers: 0

  Forensic artifact patterns found in text:
    Registry paths (\)                         366 (2.1%)
    File extensions (.exe, .dll)              1443 (8.3%)
    IP addresses                                 8 (0.0%)
    Hashes (hex 32+ chars)                       1 (0.0%)
    Command-line tools                         687 (3.9%)
    ATT&CK technique refs                       17 (0.1%)

=== SAMPLE TEXTS (5 random examples) ===

  [Defense Evasion]
  After encrypting its own log files, the log encryption module in Prikormka deletes the original, unencrypted files from the host....

  [Persistence]
  During the 2016 Ukraine Electric Power Attack, Sandworm Team used a trojanized version of Windows Notepad to add a layer of persistence for Industroyer....

  [Initial Access]
  Ajax Security Team has used personalized spearphishing attachments.

## CELL 9: Save Datasets


WHAT THIS DOES:
Saves the train/dev/test splits as CSV files.
Also saves the full dataset and a metadata summary.


In [25]:
# Columns to save
save_cols = ['text', 'label', 'technique_id', 'technique_name', 'tactics',
             'source_name', 'source_type', 'source_id', 'domain']

# Save splits
df_train[save_cols].to_csv(os.path.join(CLASS_DIR, 'train.csv'), index=False)
df_dev[save_cols].to_csv(os.path.join(CLASS_DIR, 'dev.csv'), index=False)
df_test[save_cols].to_csv(os.path.join(CLASS_DIR, 'test.csv'), index=False)
df_all[save_cols].to_csv(os.path.join(CLASS_DIR, 'full_dataset.csv'), index=False)

print(f"✓ Saved train.csv:        {len(df_train)} rows")
print(f"✓ Saved dev.csv:          {len(df_dev)} rows")
print(f"✓ Saved test.csv:         {len(df_test)} rows")
print(f"✓ Saved full_dataset.csv: {len(df_all)} rows")
print(f"  Location: {CLASS_DIR}/")

# Save metadata
metadata = {
    'total_examples': len(df_all),
    'train_size': len(df_train),
    'dev_size': len(df_dev),
    'test_size': len(df_test),
    'num_classes': df_all['label'].nunique(),
    'classes': sorted(df_all['label'].unique().tolist()),
    'split_method': 'GroupShuffleSplit by source_id (case-level)',
    'split_ratio': '70/15/15',
    'random_state': 42,
    'sources': {
        'enterprise_attack_v18.1': len(enterprise_examples),
        'mobile_attack_v18.1': len(mobile_examples),
    },
    'label_distribution': df_all['label'].value_counts().to_dict(),
}

import json
with open(os.path.join(CLASS_DIR, 'dataset_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Saved dataset_metadata.json")

✓ Saved train.csv:        12325 rows
✓ Saved dev.csv:          2409 rows
✓ Saved test.csv:         2687 rows
✓ Saved full_dataset.csv: 17421 rows
  Location: /content/drive/MyDrive/forensicnlp/data/classification/
✓ Saved dataset_metadata.json


## CELL 10: Extract Forensic Strings for Tokenization Corpus (Stage 1)


WHAT THIS DOES:
For Stage 1 (tokenization analysis), we need raw forensic strings.
We extract them from the procedure example texts using regex patterns.

This captures: file paths, registry paths, commands, IPs, hashes,
executable names, and other forensic-relevant strings embedded in
the ATT&CK procedure descriptions.

NOTE: This is a STARTING POINT. You should supplement this with strings
from EVTX-ATTACK-SAMPLES, Sigma rules, and OTRF datasets when you
download them.


In [26]:
def extract_forensic_strings(texts):
    """Extract forensic artifact strings from procedure example text."""

    strings = []

    patterns = {
        'file_path': r'[A-Z]:\\(?:[^\s,;)]+)',  # Windows paths like C:\Users\...
        'registry_path': r'(?:HKLM|HKCU|HKCR|HKU|HKCC)\\[^\s,;)]+',  # Registry paths
        'unix_path': r'/(?:etc|var|tmp|usr|bin|opt|home)/[^\s,;)]+',  # Unix paths
        'executable': r'\b\w+\.(?:exe|dll|bat|ps1|vbs|js|py|sh)\b',  # Executable files
        'ip_address': r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}(?::\d+)?\b',  # IP:port
        'hash_value': r'\b[a-fA-F0-9]{32,64}\b',  # MD5/SHA hashes
        'url': r'https?://[^\s,;)"]+',  # URLs
        'command_line': r'(?:cmd|powershell|wmic|reg|net|sc|schtasks|certutil|bitsadmin)(?:\.exe)?\s+[^\n]{10,80}',  # Commands
    }

    for text in texts:
        for category, pattern in patterns.items():
            matches = re.findall(pattern, text, re.IGNORECASE)
            for match in matches:
                match = match.strip().rstrip('.')
                if len(match) > 5:  # Skip very short matches
                    strings.append({
                        'string': match,
                        'category': category,
                        'source': 'mitre_attack'
                    })

    return strings

forensic_strings = extract_forensic_strings(df_all['text'].tolist())
df_strings = pd.DataFrame(forensic_strings).drop_duplicates(subset='string')

print(f"=== EXTRACTED FORENSIC STRINGS ===\n")
print(f"  Total unique strings: {len(df_strings)}")
print(f"\n  By category:")
for cat, count in df_strings['category'].value_counts().items():
    print(f"    {cat:20s} {count:5d}")

# Save
df_strings.to_json(
    os.path.join(TOKEN_DIR, 'forensic_strings_from_attack.jsonl'),
    orient='records', lines=True
)
print(f"\n✓ Saved to {TOKEN_DIR}/forensic_strings_from_attack.jsonl")

# Show samples
print(f"\n=== SAMPLE STRINGS ===\n")
for cat in df_strings['category'].unique():
    samples = df_strings[df_strings['category'] == cat].head(3)
    print(f"  [{cat}]")
    for _, row in samples.iterrows():
        print(f"    {row['string'][:100]}")
    print()

=== EXTRACTED FORENSIC STRINGS ===

  Total unique strings: 1735

  By category:
    command_line           983
    executable             460
    registry_path          115
    file_path              107
    unix_path               62
    ip_address               4
    url                      3
    hash_value               1

✓ Saved to /content/drive/MyDrive/forensicnlp/data/tokenization/forensic_strings_from_attack.jsonl

=== SAMPLE STRINGS ===

  [command_line]
    net settings to lower internet security
    net settings to lower internet security before launching `rundll32.exe`, which in-tu
    PowerShell scripts to run a credential harvesting tool in memory to evade defenses

  [executable]
    vba_macro.exe
    rundll32.exe
    svchost.exe

  [file_path]
    C:\ProgramData</code>
    c:\ProgramData\found_shares.txt`
    c:\|findstr

  [registry_path]
    hklm\sam
    hklm\system
    hklm\security</code>

  [url]
    http://103.224.80[.]44:8080/kernel`
    http://ip-api.com/json

## CELL 11: Summary and Next Steps


WHAT THIS DOES:
Prints a summary of everything we created and what to do next.


In [27]:
print("=" * 60)
print("  ForensicNLP — Data Extraction Complete!")
print("=" * 60)

print(f"""
FILES CREATED:
  {CLASS_DIR}/
    ├── full_dataset.csv         ({len(df_all)} labeled examples)
    ├── train.csv                ({len(df_train)} examples)
    ├── dev.csv                  ({len(df_dev)} examples)
    ├── test.csv                 ({len(df_test)} examples)
    └── dataset_metadata.json    (split info, class distribution)

  {TOKEN_DIR}/
    └── forensic_strings_from_attack.jsonl  ({len(df_strings)} unique strings)

DATASET STATS:
  Classes:       {df_all['label'].nunique()}
  Total samples: {len(df_all)}
  Domains:       Enterprise + Mobile ATT&CK v18.1
  Split method:  Case-level (no source leakage)
  Split ratio:   70/15/15

NEXT STEPS:
  1. Download supplemental sources for Stage 1 tokenization:
     git clone https://github.com/sbousseaden/EVTX-ATTACK-SAMPLES.git
     git clone https://github.com/SigmaHQ/sigma.git

  2. Run notebook 02: Tokenization Fragmentation Analysis
     → Compare BERT, RoBERTa, and custom tokenizers on forensic strings

  3. Run notebook 03: Classification Experiments
     → Fine-tune BERT, RoBERTa, ForensicRoBERTa on train.csv
     → Evaluate on test.csv with macro-F1
""")

  ForensicNLP — Data Extraction Complete!

FILES CREATED:
  /content/drive/MyDrive/forensicnlp/data/classification/
    ├── full_dataset.csv         (17421 labeled examples)
    ├── train.csv                (12325 examples)
    ├── dev.csv                  (2409 examples)
    ├── test.csv                 (2687 examples)
    └── dataset_metadata.json    (split info, class distribution)

  /content/drive/MyDrive/forensicnlp/data/tokenization/
    └── forensic_strings_from_attack.jsonl  (1735 unique strings)

DATASET STATS:
  Classes:       14
  Total samples: 17421
  Domains:       Enterprise + Mobile ATT&CK v18.1
  Split method:  Case-level (no source leakage)
  Split ratio:   70/15/15

NEXT STEPS:
  1. Download supplemental sources for Stage 1 tokenization:
     git clone https://github.com/sbousseaden/EVTX-ATTACK-SAMPLES.git
     git clone https://github.com/SigmaHQ/sigma.git

  2. Run notebook 02: Tokenization Fragmentation Analysis
     → Compare BERT, RoBERTa, and custom tokenizers